In [ ]:
import cv2
import numpy as np
import os
from tqdm import tqdm

# -----------------------------
# Paths
# -----------------------------
reference_path = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\citra_2025-12-12_164501.jpg"
folder_path = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2"
output_folder = os.path.join(folder_path, r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\down3_normalized_duct_tape")

os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# ROI (duct tape)
# -----------------------------
y1, y2 = 0, 1158
x1, x2 = 866, 943

# -----------------------------
# Load reference image
# -----------------------------
ref_img = cv2.imread(reference_path).astype(np.float32)

ref_roi = ref_img[y1:y2, x1:x2]

# Average RGB of reference duct tape
ref_mean = np.mean(ref_roi.reshape(-1,3), axis=0)

print("Reference duct tape RGB:", ref_mean)

# -----------------------------
# Process all images
# -----------------------------
for file in tqdm(os.listdir(folder_path)):

    if not file.lower().endswith((".jpg",".png",".jpeg")):
        continue

    img_path = os.path.join(folder_path, file)

    img = cv2.imread(img_path).astype(np.float32)

    # Extract duct tape ROI
    roi = img[y1:y2, x1:x2]

    unk_mean = np.mean(roi.reshape(-1,3), axis=0)

    # Scaling factors
    scale = ref_mean / unk_mean

    # Apply normalization
    normalized = img * scale

    normalized = np.clip(normalized, 0, 255).astype(np.uint8)

    # Save image
    save_path = os.path.join(output_folder, 'back_row_down2file)
    cv2.imwrite(save_path, normalized)

print("Normalization finished")

Reference duct tape RGB: [49.466866 52.204075 50.049103]


  0%|          | 0/1558 [00:00<?, ?it/s]

100%|██████████| 1558/1558 [00:28<00:00, 53.98it/s]

Normalization finished


In [13]:
import os
from PIL import Image

# input and output folders
input_folder = r"F:\strwaberry images\back row upated with text"
output_base = r"F:\strwaberry images\back row upated with text\Cropped_Images"

# cropping coordinates: (x1, y1, x2, y2)
crops = {
    "down_5": (3683, 676, 4569,1489),
    # "top_2": (310,366, 1383,907),
    # "top_4": (1917, 343, 2915,860),
    # "top_5": (2875,307, 3687, 768),
}

# create output folders
for folder in crops.keys():
    os.makedirs(os.path.join(output_base, folder), exist_ok=True)

# process images
for img_name in os.listdir(input_folder):
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(input_folder, img_name)
        image = Image.open(img_path)

        for folder_name, (x1, y1, x2, y2) in crops.items():
            cropped = image.crop((x1, y1, x2, y2))

            save_path = os.path.join(output_base, folder_name, img_name)
            cropped.save(save_path)

print("Cropping completed")


Cropping completed


In [4]:
import cv2
import numpy as np
import os
from pathlib import Path

# -------- COLOR MATCH FUNCTION --------
def color_transfer(source, target):
    source = cv2.cvtColor(source, cv2.COLOR_BGR2LAB).astype("float32")
    target = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype("float32")

    (lMeanSrc, aMeanSrc, bMeanSrc) = cv2.mean(source)[:3]
    (lStdSrc, aStdSrc, bStdSrc) = source.std(axis=(0,1))

    (lMeanTar, aMeanTar, bMeanTar) = cv2.mean(target)[:3]
    (lStdTar, aStdTar, bStdTar) = target.std(axis=(0,1))

    l, a, b = cv2.split(source)

    l = ((l - lMeanSrc) * (lStdTar / (lStdSrc+1e-6))) + lMeanTar
    a = ((a - aMeanSrc) * (aStdTar / (aStdSrc+1e-6))) + aMeanTar
    b = ((b - bMeanSrc) * (bStdTar / (bStdSrc+1e-6))) + bMeanTar

    result = cv2.merge([l, a, b])
    result = np.clip(result, 0, 255).astype("uint8")
    return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)

# -------- LOAD REFERENCE --------
reference = cv2.imread(r"F:\strwaberry images\back row upated with text\Cropped_Images\top_5\citra_2025-12-28_163001.jpg")

input_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\top_5"          
output_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\top_5\preprocessed_images_lighting_removal"  
os.makedirs(output_folder, exist_ok=True)

# -------- PROCESS ALL IMAGES --------
for img_path in Path(input_folder).glob("*"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    normalized = color_transfer(img, reference)
    cv2.imwrite(os.path.join(output_folder, 'back_row_top5'+img_path.name), normalized)

print("✔ Dataset normalized to reference lighting")

✔ Dataset normalized to reference lighting


In [7]:
!pip install transparent-background --user

  Using cached transparent_background-1.3.4-py3-none-any.whl.metadata (18 kB)
  Using cached timm-1.0.25-py3-none-any.whl.metadata (38 kB)
  Using cached kornia-0.8.2-py2.py3-none-any.whl.metadata (18 kB)
  Using cached gdown-5.2.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached albucore-0.0.41-py3-none-any.whl.metadata (10 kB)
  Using cached pymatting-1.1.15-py3-none-any.whl.metadata (8.7 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached kornia_rs-0.1.10-cp312-cp312-win_amd64.whl.metadata (13 kB)
Using cached transparent_background-1.3.4-py3-none-any.whl (33 kB)
Using cached albumentations-2.0.8-py3-none-any.whl (369 kB)
Using cached albucore-0.0.24-py3-none-any.whl (15 kB)
Using cached gdown-5.2.1-py3-none-any.whl (18 kB)
Using cached kornia-0.8.2-py2.py3-none-any.whl (1.1 MB)
Using cached k

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import os
import cv2
import numpy as np
from transparent_background import Remover

# =========================
# INPUT / OUTPUT PATHS
# =========================
input_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\top_2\preprocessed_images_lighting_removal"
output_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\top_2\preprocessed_images_lighting_removal\soil_removal"

os.makedirs(output_folder, exist_ok=True)

# =========================
# CROP COORDINATES
# =========================
# (left-top) (right-bottom)
x1, y1, x2, y2 = 0,0, 1072, 538

# =========================
# INIT REMOVER (LOAD ONCE)
# =========================
remover = Remover()

# =========================
# PROCESS ALL IMAGES
# =========================
valid_ext = (".png", ".jpg", ".jpeg")

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(valid_ext):
        continue

    img_path = os.path.join(input_folder, filename)
    print(f"Processing: {filename}")

    # -----------------------
    # LOAD IMAGE
    # -----------------------
    img = cv2.imread(img_path)

    if img is None:
        print(f"Skipping (cannot read): {filename}")
        continue

    h, w = img.shape[:2]

    # -----------------------
    # SAFE COORDINATE CHECK
    # -----------------------
    x1_c = max(0, min(x1, w))
    x2_c = max(0, min(x2, w))
    y1_c = max(0, min(y1, h))
    y2_c = max(0, min(y2, h))

    if x2_c <= x1_c or y2_c <= y1_c:
        print(f"Invalid crop for {filename}, skipping...")
        continue

    # -----------------------
    # CROP
    # -----------------------
    crop = img[y1_c:y2_c, x1_c:x2_c]

    # -----------------------
    # REMOVE BACKGROUND
    # -----------------------
    rgba = remover.process(crop, type="rgba")
    mask = rgba[:, :, 3]

    # -----------------------
    # CLEAN MASK
    # -----------------------
    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)

    # -----------------------
    # APPLY MASK
    # -----------------------
    plant = cv2.bitwise_and(crop, crop, mask=mask)

    # -----------------------
    # TIGHT CROP
    # -----------------------
    coords = cv2.findNonZero(mask)

    if coords is not None:
        x, y, w_box, h_box = cv2.boundingRect(coords)
        final = plant[y:y+h_box, x:x+w_box]
    else:
        print(f"No plant detected in {filename}, saving original crop")
        final = crop

    # -----------------------
    # SAVE OUTPUT
    # -----------------------
    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, final)

print("✅ Processing Complete!")

C:\Users\rohan\AppData\Roaming\Python\Python312\site-packages\transparent_background\gui.py:24: UserWarning: Failed to import flet. Ignore this message when you do not need GUI mode.
  warnings.warn('Failed to import flet. Ignore this message when you do not need GUI mode.')
c:\Users\rohan\anaconda3\envs\aml\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Settings -> Mode=base, Device=cpu, Torchscript=disabled
Processing: back_row_top2citra_2025-12-12_162047.jpg
Processing: back_row_top2citra_2025-12-12_163001.jpg
Processing: back_row_top2citra_2025-12-12_164501.jpg
Processing: back_row_top2citra_2025-12-12_170001.jpg


KeyboardInterrupt: 

In [22]:
import os
import shutil

# =========================
# PATHS (EDIT HERE)
# =========================
input_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\top_5\preprocessed_images_lighting_removal"
output_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\top_5\preprocessed_images_lighting_removal\output_lightning_normalization"

os.makedirs(output_folder, exist_ok=True)

# =========================
# PREFIX TO REMOVE
# =========================
prefix = "back_row_top5"

# =========================
# PROCESS FILES
# =========================
valid_ext = (".png", ".jpg", ".jpeg")

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(valid_ext):
        continue

    old_path = os.path.join(input_folder, filename)

    # -------------------------
    # REMOVE PREFIX
    # -------------------------
    if filename.startswith(prefix):
        new_name = filename[len(prefix):]
    else:
        # if prefix not found → keep same name
        new_name = filename

    new_path = os.path.join(output_folder, new_name)

    # -------------------------
    # COPY FILE (safe)
    # -------------------------
    shutil.copy2(old_path, new_path)

    print(f"{filename} → {new_name}")

print(" Done renaming and saving to new folder!")

back_row_top5citra_2025-12-12_162047.jpg → citra_2025-12-12_162047.jpg
back_row_top5citra_2025-12-12_163001.jpg → citra_2025-12-12_163001.jpg
back_row_top5citra_2025-12-12_164501.jpg → citra_2025-12-12_164501.jpg
back_row_top5citra_2025-12-12_170001.jpg → citra_2025-12-12_170001.jpg
back_row_top5citra_2025-12-12_171501.jpg → citra_2025-12-12_171501.jpg
back_row_top5citra_2025-12-13_063001.jpg → citra_2025-12-13_063001.jpg
back_row_top5citra_2025-12-13_064501.jpg → citra_2025-12-13_064501.jpg
back_row_top5citra_2025-12-13_070001.jpg → citra_2025-12-13_070001.jpg
back_row_top5citra_2025-12-13_071501.jpg → citra_2025-12-13_071501.jpg
back_row_top5citra_2025-12-13_073001.jpg → citra_2025-12-13_073001.jpg
back_row_top5citra_2025-12-13_074501.jpg → citra_2025-12-13_074501.jpg
back_row_top5citra_2025-12-13_080001.jpg → citra_2025-12-13_080001.jpg
back_row_top5citra_2025-12-13_081501.jpg → citra_2025-12-13_081501.jpg
back_row_top5citra_2025-12-13_083001.jpg → citra_2025-12-13_083001.jpg
back_r

In [24]:
import os
import pandas as pd

# =========================
# PATHS (EDIT HERE)
# =========================
excel_path = r"Back_row_excel_dataset\final_dataset_back_setup_down_plant2.xlsx"
image_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\top_5\preprocessed_images_lighting_removal"

# =========================
# LOAD EXCEL
# =========================
df = pd.read_excel(excel_path)

# -------------------------
# DROP COLUMN
# -------------------------
if "green_pixel" in df.columns:
    df = df.drop(columns=["green_pixel"])
    print("✅ Dropped 'green_pixel' column")

# -------------------------
# GET IMAGE NAMES FROM EXCEL
# -------------------------
# Change this if your column name is different
excel_image_col = "Image_name"

if excel_image_col not in df.columns:
    raise ValueError(f"Column '{excel_image_col}' not found in Excel")

excel_images = set(df[excel_image_col].astype(str))

# -------------------------
# GET IMAGE FILES FROM FOLDER
# -------------------------
valid_ext = (".png", ".jpg", ".jpeg")

folder_images = set([
    f for f in os.listdir(image_folder)
    if f.lower().endswith(valid_ext)
])

# =========================
# COMPARISON
# =========================

# Excel → Missing in folder
missing_in_folder = excel_images - folder_images

# Folder → Not in Excel
extra_in_folder = folder_images - excel_images

# =========================
# RESULTS
# =========================

print("\n SUMMARY")
print(f"Total Excel images: {len(excel_images)}")
print(f"Total Folder images: {len(folder_images)}")

print("\n Missing images in folder (present in Excel):")
for img in list(missing_in_folder)[:20]:
    print(img)

print(f"Total missing: {len(missing_in_folder)}")

print("\n Extra images in folder (not in Excel):")
for img in list(extra_in_folder)[:20]:
    print(img)

print(f"Total extra: {len(extra_in_folder)}")


 SUMMARY
Total Excel images: 1558
Total Folder images: 1560

 Missing images in folder (present in Excel):
nan
Total missing: 1

 Extra images in folder (not in Excel):
citra_2025-12-12_163001.jpg
citra_2025-12-12_162047.jpg
newplot.png
Total extra: 3


In [25]:
import os
import pandas as pd

# =========================
# PATHS (EDIT HERE)
# =========================
# excel_path = r"YOUR_EXCEL_PATH.xlsx"
# image_folder = r"YOUR_IMAGE_FOLDER_PATH"
output_excel_path = r"final_dataset_back_setup_top_plant5.xlsx"

# =========================
# SETTINGS
# =========================
image_col = "Image_name"   # change if needed
prefix = "Top5_"      # your prefix

# =========================
# LOAD EXCEL
# =========================
df = pd.read_excel(excel_path)

# Drop column if exists
if "green_pixel" in df.columns:
    df = df.drop(columns=["green_pixel"])

# =========================
# GET IMAGE LISTS
# =========================
excel_images = set(df[image_col].astype(str))

valid_ext = (".png", ".jpg", ".jpeg")
folder_images = set([
    f for f in os.listdir(image_folder)
    if f.lower().endswith(valid_ext)
])

# =========================
# FIND VALID MATCHES
# =========================
valid_images = excel_images.intersection(folder_images)

print(f"✅ Valid matches: {len(valid_images)}")

# =========================
# 1️⃣ RENAME IMAGES (ADD PREFIX)
# =========================
for img in valid_images:
    old_path = os.path.join(image_folder, img)
    new_name = prefix + img
    new_path = os.path.join(image_folder, new_name)

    # avoid overwrite
    if not os.path.exists(new_path):
        os.rename(old_path, new_path)

print("✅ Prefix added to images")

# =========================
# 2️⃣ CLEAN EXCEL (REMOVE EXTRA ROWS)
# =========================
df_cleaned = df[df[image_col].isin(valid_images)].copy()

# ALSO update names in excel (add prefix)
df_cleaned[image_col] = prefix + df_cleaned[image_col]

# =========================
# SAVE CLEANED EXCEL
# =========================
df_cleaned.to_excel(output_excel_path, index=False)

print("✅ Cleaned Excel saved")
print(f"Final rows: {len(df_cleaned)}")

✅ Valid matches: 1557
✅ Prefix added to images
✅ Cleaned Excel saved
Final rows: 1557


In [26]:
import pandas as pd

# =========================
# PATHS (EDIT HERE)
# =========================
excel_path = r"F:\strwaberry images\Back_row_excel_dataset\final_dataset_back_setup_top_plant2.xlsx"
output_excel_path = r"final_dataset_back_setup_top_plant2.xlsx"

# =========================
# SETTINGS
# =========================
image_col = "Image_name"   # change if needed
old_prefix = "Down2"

# OPTION 1: Replace with new prefix
new_prefix = "Top2"   # "" → removes prefix
# new_prefix = "processed_"  # uncomment if you want replacement

# =========================
# LOAD EXCEL
# =========================
df = pd.read_excel(excel_path)

if image_col not in df.columns:
    raise ValueError(f"{image_col} not found in Excel")

# =========================
# REPLACE PREFIX FUNCTION
# =========================
def replace_prefix(name):
    name = str(name)
    if name.startswith(old_prefix):
        return new_prefix + name[len(old_prefix):]
    return name

# =========================
# APPLY TRANSFORMATION
# =========================
df[image_col] = df[image_col].apply(replace_prefix)

# =========================
# SAVE OUTPUT
# =========================
df.to_excel(output_excel_path, index=False)

print("✅ Excel updated successfully!")

✅ Excel updated successfully!


In [27]:
df=pd.read_excel(r"F:\strwaberry images\Back_row_excel_dataset\final_dataset_back_setup_down_plant2.xlsx")

df.dropna(subset=["Image_name"], inplace=True)

In [29]:
df.to_excel(r"final_dataset_back_setup_down_plant2.xlsx", index=False)